# Clonal Dynamics (Cell + Nucleus)

For each subclass (dying, persisting, slow proliferator, fast
proliferator), finds the 3 `vem_1uM` clones (lineages, identified by
`lin_id`) with the most TRUE segmented cells - among those with enough
cells in the CAJAL-embedded sample to actually plot - and paints each
one's own cells onto the shared embedding, in its own color, against the
whole vem population in light grey. One stacked, 3-panel figure per
subclass (same layout as Figure 1g of Shaffer et al., "Diverse clonal
fates emerge upon drug treatment of homogeneous cancer cells").

See `shared/clonal_dynamics.py`'s own docstrings (`pick_top_clones`,
`plot_clones_on_embedding`, `export_clones_by_subclass`) for exactly how
a clone is selected and plotted.

Plots both embeddings `main.py` computes (UMAP and PHATE) every time
this notebook runs, no manual switch needed.

Requires `main.py` to have been run first.

In [ ]:
import os
import sys

import pandas as pd

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/01_naive_vs_vem/notebooks/), so the analysis dir - and config.py -
# is one level up, and the repo root - where shared/ lives - is two more
# levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset
from shared.classify import classify_subclass
from shared.clonal_dynamics import export_clones_by_subclass

## Load the true vem population

The full `vem_1uM_metadata.csv` (restricted to `has_mask=True`),
independent of modality or embedding method - classified into the same
4 subclasses `select_cells.py` already uses. This is the TRUE population
each clone's cell count is measured against in `pick_top_clones`, not
just the randomly budgeted subset that went through CAJAL (`main.py`'s
own `selected_metadata.csv`).

In [ ]:
vem_metadata = pd.read_csv(config.metadata_csv_path(config.VEM_CONDITION))
vem_metadata = vem_metadata[vem_metadata['has_mask']].copy()
vem_metadata['subclass'] = classify_subclass(
    vem_metadata,
    dying_m_label=config.DYING_M_LABEL, persisting_m_label=config.PERSISTING_M_LABEL,
    prolif_m_label=config.PROLIF_M_LABEL,
    slow_k_labels=config.SLOW_K_LABELS, fast_k_labels=config.FAST_K_LABELS,
)
print(f"{len(vem_metadata)} true has_mask vem_1uM cells across {vem_metadata['lin_id'].nunique()} lineages")

## Plot: top 3 clones per subclass, painted onto the embedding

For each embedding method, one PDF, one SVG per subclass, written to
`output/plots/06_clonal_dynamics/` (SVGs in their own `svgs/` subfolder
there), plus one `{modality}_{embedding_method}_clone_summary.csv`
recording exactly which lin_ids were chosen and why (true/embedded cell
counts, embedded frame-span).

In [ ]:
N_CLONES = 3
MIN_EMBEDDED_CELLS = 5   # a clone needs at least this many cells in the CAJAL-embedded sample to be worth plotting

NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '06_clonal_dynamics')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    For each embedding method, load this modality's vem-only embedding
    and paint the top N_CLONES clones per subclass onto it.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        vem_dfs: {embedding_method: the vem-only embedded subset for this
                  modality and embedding method} (in-memory only)
    '''
    vem_dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        vem_df = df[df['condition'] == config.VEM_CONDITION]
        print(f"[{modality}/{embedding_method}] {len(vem_df)} embedded vem cells across "
              f"{vem_df['subclass'].nunique()} subclasses")

        export_clones_by_subclass(
            vem_df, vem_metadata, NOTEBOOK_PLOTS_DIR, f'{modality}_{embedding_method}', embedding_method.upper(),
            title_prefix=f'{MODALITY_NAMES[modality]} morphology',
            n_clones=N_CLONES, min_embedded_cells=MIN_EMBEDDED_CELLS,
        )
        vem_dfs[embedding_method] = vem_df

    return vem_dfs

## Run: cell morphology

In [ ]:
cell_vem_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_vem_dfs = process_modality('nuc')
